# Qwen3-VL 4B Colab Runtime

Local test runtime for HCMVision. Select `Runtime > Change runtime type > T4 GPU`, then run cells from top to bottom.

In [ ]:
# Fail early if Colab is not using a GPU runtime.
!nvidia-smi

In [ ]:
# Qwen3-VL support is in recent Transformers; install from source for Colab.
!pip -q uninstall -y pillow PIL
!pip -q install --no-cache-dir --force-reinstall "pillow==11.3.0"
!pip -q install -U git+https://github.com/huggingface/transformers accelerate bitsandbytes qwen-vl-utils fastapi uvicorn pyngrok python-multipart

In [ ]:
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

# Use the same value in HCMVision.Backend/appsettings.Local.json -> AI:RemoteQwen:ApiToken.
API_TOKEN = "change-me-local-ai-token"

# Optional but recommended. Create one at https://dashboard.ngrok.com/get-started/your-authtoken.
NGROK_AUTHTOKEN = ""

MAX_IMAGE_SIDE = 768
MAX_NEW_TOKENS = 220

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime is required. Select Runtime > Change runtime type > T4 GPU.")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.eval()
print("Loaded", MODEL_ID)

In [ ]:
import json
import os
import re
import tempfile
import threading
import time
from typing import Optional

import uvicorn
from fastapi import FastAPI, File, Header, HTTPException, UploadFile
from PIL import Image
from pyngrok import ngrok
from qwen_vl_utils import process_vision_info

app = FastAPI(title="HCMVision Qwen3-VL Runtime")

PROMPT = """
Analyze this Ho Chi Minh City traffic camera image.
Return only valid JSON, no markdown, no extra text.
Classify visible rain and traffic condition from the image.
Use exact enum values:
- rain_level: none, light, medium, heavy
- traffic_level: clear, slow, jam, unknown
- confidence: number from 0.0 to 1.0
- reason: short Vietnamese or English reason
JSON schema:
{\"rain_level\":\"none|light|medium|heavy\",\"traffic_level\":\"clear|slow|jam|unknown\",\"confidence\":0.0,\"reason\":\"short reason\"}
""".strip()

def check_token(x_ai_token: Optional[str]) -> None:
    if API_TOKEN and x_ai_token != API_TOKEN:
        raise HTTPException(status_code=401, detail="Invalid AI token")

def resize_image(image: Image.Image) -> Image.Image:
    image = image.convert("RGB")
    image.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE))
    return image

def extract_json(text: str) -> dict:
    text = text.strip()
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError(f"Model did not return JSON: {text[:500]}")
    return json.loads(match.group(0))

def normalize_rain(value) -> str:
    value = str(value or "").strip().lower()
    if value in {"heavy", "storm", "severe", "high"}:
        return "heavy"
    if value in {"medium", "moderate"}:
        return "medium"
    if value in {"light", "drizzle", "low"}:
        return "light"
    return "none"

def normalize_traffic(value) -> str:
    value = str(value or "").strip().lower()
    if value in {"clear", "free", "free_flow", "normal"}:
        return "clear"
    if value in {"slow", "moderate", "busy"}:
        return "slow"
    if value in {"jam", "congested", "heavy", "blocked"}:
        return "jam"
    return "unknown"

def normalize_confidence(value) -> float:
    try:
        score = float(value)
    except Exception:
        score = 0.0
    if score > 1:
        score = score / 100.0
    return max(0.0, min(1.0, score))

def predict_image(image: Image.Image) -> dict:
    image = resize_image(image)
    tmp_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
            image.save(tmp.name, format="JPEG", quality=90)
            tmp_path = tmp.name

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": tmp_path},
                    {"type": "text", "text": PROMPT},
                ],
            }
        ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to("cuda")

        with torch.inference_mode():
            generated_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)

        generated_ids_trimmed = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
        raw = extract_json(output_text)
        return {
            "rain_level": normalize_rain(raw.get("rain_level") or raw.get("rainLevel")),
            "traffic_level": normalize_traffic(raw.get("traffic_level") or raw.get("trafficLevel")),
            "confidence": normalize_confidence(raw.get("confidence", 0.0)),
            "reason": str(raw.get("reason") or "")[:300],
            "model": MODEL_ID,
        }
    finally:
        if tmp_path and os.path.exists(tmp_path):
            os.remove(tmp_path)

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID, "gpu": torch.cuda.get_device_name(0)}

@app.post("/predict")
async def predict(image: UploadFile = File(...), x_ai_token: Optional[str] = Header(default=None)):
    check_token(x_ai_token)
    try:
        pil_image = Image.open(image.file)
        return predict_image(pil_image)
    except HTTPException:
        raise
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc)) from exc

if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()
public_url = ngrok.connect(8000, "http").public_url
print("Public URL:", public_url)
print("Backend BaseUrl should be:", public_url)
print("Health:", public_url + "/health")

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)
print("Runtime ready")